<a href="https://colab.research.google.com/github/kotwalurja-03/Google-Collab-Project/blob/main/ShopLens_%E2%80%94_SQL_%26_Python_Sales_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install plotly -q

import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("All libraries ready!")


All libraries ready!


In [ ]:
conn = sqlite3.connect("ecommerce.db")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE IF NOT EXISTS customers (
  customer_id INTEGER PRIMARY KEY,
  name TEXT,
  city TEXT,
  segment TEXT
);

CREATE TABLE IF NOT EXISTS products (
  product_id INTEGER PRIMARY KEY,
  name TEXT,
  category TEXT,
  price REAL
);

CREATE TABLE IF NOT EXISTS orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER,
  product_id INTEGER,
  quantity INTEGER,
  order_date TEXT,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
  FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

customers = [
  (1,"Arjun Shah","Mumbai","Retail"),
  (2,"Priya Mehta","Delhi","Wholesale"),
  (3,"Rahul Joshi","Bangalore","Retail"),
  (4,"Sneha Patel","Surat","Retail"),
  (5,"Vikram Nair","Chennai","Wholesale"),
]

products = [
  (1,"Laptop","Electronics",55000),
  (2,"Phone","Electronics",22000),
  (3,"Desk Chair","Furniture",8500),
  (4,"Notebook","Stationery",150),
  (5,"Headphones","Electronics",3500),
  (6,"Pen Set","Stationery",200),
  (7,"Monitor","Electronics",18000),
]

orders = [
  (1,1,1,2,"2024-01-05"),(2,2,3,5,"2024-01-12"),
  (3,3,2,1,"2024-01-20"),(4,4,4,10,"2024-02-03"),
  (5,1,5,3,"2024-02-14"),(6,5,7,2,"2024-02-22"),
  (7,2,1,1,"2024-03-01"),(8,3,6,20,"2024-03-10"),
  (9,4,2,2,"2024-03-18"),(10,5,3,3,"2024-03-25"),
  (11,1,7,1,"2024-04-02"),(12,2,5,4,"2024-04-11"),
  (13,3,4,15,"2024-04-19"),(14,4,1,1,"2024-05-05"),
  (15,5,2,3,"2024-05-14"),
]

cur.executemany("INSERT OR IGNORE INTO customers VALUES (?,?,?,?)", customers)
cur.executemany("INSERT OR IGNORE INTO products VALUES (?,?,?,?)", products)
cur.executemany("INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?)", orders)
conn.commit()
print("Database created with", cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0], "orders")


Database created with 15 orders


In [ ]:

q1 = """
SELECT p.category,
       SUM(o.quantity * p.price) AS total_revenue,
       COUNT(o.order_id) AS total_orders
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
"""
df_category = pd.read_sql_query(q1, conn)
df_category



,category,total_revenue,total_orders
0,Electronics,430500.0,10
1,Furniture,68000.0,2
2,Stationery,7750.0,3


In [ ]:
q2 = """
SELECT strftime('%Y-%m', o.order_date) AS month,
       SUM(o.quantity * p.price) AS revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY month
ORDER BY month
"""
df_monthly = pd.read_sql_query(q2, conn)
df_monthly


,month,revenue
0,2024-01,174500.0
1,2024-02,48000.0
2,2024-03,128500.0
3,2024-04,34250.0
4,2024-05,121000.0


In [ ]:
q3 = """
SELECT c.name, c.city, c.segment,
       SUM(o.quantity * p.price) AS total_spend
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.customer_id
ORDER BY total_spend DESC
"""
df_customers = pd.read_sql_query(q3, conn)
df_customers


,name,city,segment,total_spend
0,Arjun Shah,Mumbai,Retail,138500.0
1,Vikram Nair,Chennai,Wholesale,127500.0
2,Priya Mehta,Delhi,Wholesale,111500.0
3,Sneha Patel,Surat,Retail,100500.0
4,Rahul Joshi,Bangalore,Retail,28250.0


In [ ]:
fig1 = px.bar(
    df_category,
    x="category", y="total_revenue",
    color="category",
    title="Total Revenue by Product Category",
    labels={"total_revenue": "Revenue (₹)", "category": "Category"},
    text_auto=True
)
fig1.update_layout(showlegend=False)
fig1.show()

In [ ]:
fig2 = px.line(
    df_monthly,
    x="month", y="revenue",
    title="Monthly Revenue Trend",
    labels={"revenue": "Revenue (₹)", "month": "Month"},
    markers=True
)
fig2.update_traces(line_color="#378ADD", marker_size=8)
fig2.show()

In [ ]:
fig3 = px.bar(
    df_customers,
    x="total_spend", y="name",
    orientation="h",
    color="segment",
    title="Top Customers by Total Spend",
    labels={"total_spend": "Total Spend (₹)", "name": "Customer"},
    text_auto=True
)
fig3.update_layout(yaxis={"categoryorder": "total ascending"})
fig3.show()


In [ ]:

conn.close()
print("Connection closed.")

Connection closed.
